# SFL Scientific – Weather Station Latitude Prediction
## Executive Summary

**Problem:** Weather station records for post-2020 data have had their latitude corrupted to `-9999`. I am given pre-2021 labeled data (PS1) and must: (1) train a model to predict latitude from weather measurements, and (2) apply it to post-2020 stations (PS2) while flagging stations that are geographically outside the PS1 training region.

**Approach:**
- **Feature Engineering:** Each station's time-series is aggregated into per-station statistical summaries (mean, std, percentiles) for temperature, humidity, pressure, wind, and radiation. The core climatological signals that vary with latitude. Seasonal patterns (monthly averages) are also extracted since they capture the relationship between latitude and sun angle.
- **Model:** Random Forest Regressor is chosen for robustness to outliers (many `-9999` sentinel values replaced with NaN), interpretability, and strong performance on tabular data with moderate sample sizes. Gradient Boosting (XGBoost/LightGBM) would be a natural next step.
- **Anomaly Detection:** For out-of-distribution (OOD) stations, we use the Mahalanobis distance of a station's feature vector from the PS1 training distribution. Stations beyond a threshold are flagged as likely outside the original geography.
- **Validation:** Cross-validation k-fold across stations is used to estimate true generalization performance.

**Key Findings:**
- Temperature mean and seasonal temperature swing are the strongest predictors of latitude.
- Pressure and dew point also carry strong latitudinal signal.
- Several PS2 stations are flagged as OOD (geographically novel) based on Mahalanobis distance.

**Limitations & Next Steps:**
- A production pipeline is adjusted to ingest all station files.
- Deep learning sequence models (LSTM/Transformer) over raw time-series could improve accuracy.
- Ensemble of Random Forest + Gradient Boosting would reduce variance.

---

## 1. Imports & Configuration

In [3]:
# Standard library
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Data wrangling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Machine learning (scikit-learn: Pedregosa et al., 2011)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from scipy.spatial.distance import mahalanobis

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Sentinel value used in the raw data for missing readings
MISSING_SENTINEL = -9999

# Paths — adjust to your folder layout

PS1_DIR = r"data\PS1\*.csv"  #labeled training stations
PS2_DIR = r"data\PS2\*.csv"  # unlabeled prediction stations
OUTPUT_FILE = 'prediction_results.csv'

print('Environment ready.')

Environment ready.


## 2. Data Loading & Exploratory Data Analysis

In [6]:
# ── Column name mapping (Portuguese → short English aliases) ──────────────────
COL_MAP = {
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)':                    'precip',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)': 'pressure',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)':     'pressure_max',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)':    'pressure_min',
    'RADIACAO GLOBAL (Kj/m²)':                             'radiation',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)':        'temp',
    'TEMPERATURA DO PONTO DE ORVALHO (°C)':                'dew_point',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)':          'temp_max',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)':          'temp_min',
    'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)':    'dew_max',
    'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)':    'dew_min',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)':            'humidity_max',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)':            'humidity_min',
    'UMIDADE RELATIVA DO AR, HORARIA (%)':                 'humidity',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))':                'wind_dir',
    'VENTO, RAJADA MAXIMA (m/s)':                          'wind_gust',
    'VENTO, VELOCIDADE HORARIA (m/s)':                     'wind_speed',
}

# Numeric measurement columns (used for feature engineering)
MEASUREMENT_COLS = list(COL_MAP.values())


def load_station_file(filepath: str) -> pd.DataFrame:
    """
    Load a single station CSV, standardize column names,
    parse datetime, and replace sentinel -9999 with NaN.
    """
    df = pd.read_csv(filepath, index_col=0)
    
    # Rename Portuguese columns to short aliases
    df.rename(columns=COL_MAP, inplace=True)
    
    # Parse datetime
    df['datetime'] = pd.to_datetime(
        df['Data'].astype(str) + ' ' + df['Hora'].astype(str),
        errors='coerce'
    )
    df['month'] = df['datetime'].dt.month
    df['hour']  = df['datetime'].dt.hour
    
    # Replace sentinel missing values with NaN
    df[MEASUREMENT_COLS] = df[MEASUREMENT_COLS].replace(MISSING_SENTINEL, np.nan)
    
    return df


# ── Load the sample station (PS1 demo with known latitude) ───────────────────
sample_path = r"data\PS1\file_2002-2020_0.03502.csv"
df_sample = load_station_file(sample_path)

print(f'Loaded {len(df_sample)} rows | Station: {df_sample["station"].iloc[0]} '
      f'| Latitude: {df_sample["latitude"].iloc[0]}')
df_sample[MEASUREMENT_COLS].describe().round(2)

Loaded 2196 rows | Station: MACAPA | Latitude: 0.03502


,precip,pressure,pressure_max,pressure_min,radiation,temp,dew_point,temp_max,temp_min,dew_max,dew_min,humidity_max,humidity_min,humidity,wind_dir,wind_gust,wind_speed
count,1741.00,1927.00,1927.00,1927.00,975.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1714.00,1709.00,1713.00
mean,0.32,1009.96,1010.26,1009.66,1700.77,27.52,23.04,28.01,26.99,23.49,22.64,80.62,75.58,77.93,141.14,5.15,1.98
std,2.14,1.97,2.01,1.94,1038.07,2.67,0.93,2.84,2.51,0.89,1.05,12.21,14.11,13.20,82.01,2.39,1.11
min,0.00,1003.60,1003.90,1003.60,0.00,22.10,16.10,22.30,21.50,18.00,14.60,41.00,34.00,39.00,1.00,0.90,0.20
25%,0.00,1008.60,1008.80,1008.30,934.00,25.20,22.55,25.50,24.90,23.00,22.10,72.00,64.00,67.00,76.00,3.10,1.10
50%,0.00,1010.00,1010.40,1009.70,1449.00,27.10,23.10,27.70,26.50,23.50,22.80,84.00,78.00,80.00,139.00,4.90,1.70
75%,0.00,1011.40,1011.70,1011.00,2332.00,29.65,23.60,30.30,28.70,24.10,23.30,92.00,88.00,90.00,193.75,7.00,2.80
max,51.60,1015.30,1015.50,1015.00,3799.00,34.00,26.30,34.20,32.80,26.30,24.90,96.00,96.00,96.00,360.00,13.00,5.80


In [ ]:
# ── EDA: Temporal patterns for the sample station ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f'Station {df_sample["station"].iloc[0]} '
    f'(lat={df_sample["latitude"].iloc[0]:.4f}°) — Hourly Climatology',
    fontsize=13
)

monthly_means = df_sample.groupby('month')[['temp', 'humidity', 'pressure',
                                             'radiation', 'dew_point', 'wind_speed']].mean()

plot_vars = [
    ('temp',       'Temperature (°C)',       'tab:red'),
    ('humidity',   'Relative Humidity (%)',  'tab:blue'),
    ('pressure',   'Pressure (mB)',          'tab:green'),
    ('radiation',  'Solar Radiation (Kj/m²)','tab:orange'),
    ('dew_point',  'Dew Point (°C)',         'tab:purple'),
    ('wind_speed', 'Wind Speed (m/s)',       'tab:brown'),
]

for ax, (col, label, color) in zip(axes.flat, plot_vars):
    monthly_means[col].plot(ax=ax, color=color, marker='o', linewidth=2)
    ax.set_xlabel('Month')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xticks(range(1, 13))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_monthly_climatology.png', dpi=120)
plt.show()
print('Near-equatorial station (lat≈0): note minimal seasonal temperature swing — '
      'a key feature that encodes latitude.')

In [ ]:
# ── Missing data audit ────────────────────────────────────────────────────────
missing_pct = df_sample[MEASUREMENT_COLS].isna().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 4))
missing_pct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='k')
ax.set_title('Missing Data by Column (% of total rows)')
ax.set_ylabel('Missing (%)')
ax.axhline(50, color='red', linestyle='--', label='50% threshold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('eda_missing_data.png', dpi=120)
plt.show()
# Columns with >50% missing will be dropped in feature engineering
print(missing_pct.round(1).to_string())

## 3. Feature Engineering

**Rationale:** Latitude is a static property of a station, but we only have time-series measurements. We derive static feature vectors per station by aggregating the time-series into climatological statistics:

- **Global statistics** (mean, std, median, IQR, 10th/90th percentile) for each variable — capture the baseline climate.
- **Monthly means** for temperature and radiation — capture the seasonal cycle, which is directly tied to the sun's declination and thus latitude. At the equator the seasonal swing is near-zero; it grows toward the poles.
- **Diurnal range** (mean daily max − min) for temperature — also encodes climate regime.
- **Data completeness** — fraction of non-missing rows (a proxy for instrument quality / time period).

In [ ]:
# Columns to drop if missing fraction exceeds threshold
MISSING_DROP_THRESHOLD = 0.70  # drop column if >70% missing


def engineer_features(df: pd.DataFrame) -> dict:
    """
    Aggregate a station's time-series into a flat feature dictionary.

    Parameters
    ----------
    df : DataFrame — single-station time-series (already sentinel-cleaned)

    Returns
    -------
    dict of scalar features
    """
    feats = {}

    # ── Global statistics ────────────────────────────────────────────────────
    for col in MEASUREMENT_COLS:
        series = df[col].dropna()
        if len(series) == 0:
            feats[f'{col}_mean']    = np.nan
            feats[f'{col}_std']     = np.nan
            feats[f'{col}_p10']     = np.nan
            feats[f'{col}_p90']     = np.nan
        else:
            feats[f'{col}_mean']    = series.mean()
            feats[f'{col}_std']     = series.std()
            feats[f'{col}_p10']     = series.quantile(0.10)
            feats[f'{col}_p90']     = series.quantile(0.90)

    # ── Seasonal amplitude features (key for latitude) ───────────────────────
    # Monthly mean temperature — encodes seasonal cycle amplitude
    monthly_temp = df.groupby('month')['temp'].mean()
    feats['temp_seasonal_range'] = monthly_temp.max() - monthly_temp.min()
    feats['temp_seasonal_std']   = monthly_temp.std()

    # Monthly mean radiation
    monthly_rad = df.groupby('month')['radiation'].mean()
    feats['radiation_seasonal_range'] = monthly_rad.max() - monthly_rad.min()

    # Diurnal temperature range (daily max − min)
    if df['datetime'].notna().any():
        df_tmp = df[['datetime', 'temp']].dropna()
        df_tmp['date'] = df_tmp['datetime'].dt.date
        daily_range = df_tmp.groupby('date')['temp'].agg(lambda x: x.max() - x.min())
        feats['temp_diurnal_range_mean'] = daily_range.mean()
    else:
        feats['temp_diurnal_range_mean'] = np.nan

    # Monthly means for each variable (flattened)
    for col in ['temp', 'humidity', 'radiation', 'pressure', 'dew_point']:
        monthly = df.groupby('month')[col].mean()
        for m in range(1, 13):
            feats[f'{col}_m{m:02d}'] = monthly.get(m, np.nan)

    # ── Metadata ─────────────────────────────────────────────────────────────
    feats['height']              = df['height'].iloc[0]
    feats['data_completeness']   = df[MEASUREMENT_COLS].notna().mean().mean()
    feats['station_code']        = df['station_code'].iloc[0]
    feats['latitude']            = df['latitude'].iloc[0]   # target (NaN for PS2)
    feats['longitude']           = df['longitude'].iloc[0]

    return feats


# Quick test on the sample file
sample_feats = engineer_features(df_sample)
print(f"Feature vector length: {len(sample_feats)}")
print(f"Seasonal temperature range: {sample_feats['temp_seasonal_range']:.2f}°C  "
      f"(near-equatorial → expect ~0)")

## 4. Build Feature Matrix from All PS1 Stations

In [ ]:
def load_all_stations(data_dir: str, is_ps2: bool = False) -> pd.DataFrame:
    """
    Load all CSV station files from a directory, engineer features,
    and return a DataFrame with one row per station.

    Parameters
    ----------
    data_dir : str — folder containing station CSV files
    is_ps2   : bool — if True, latitude is expected to be -9999 (unknown)

    Returns
    -------
    pd.DataFrame with one row per station
    """
    files = glob.glob(os.path.join(data_dir, '*.csv'))
    if not files:
        print(f'WARNING: No CSV files found in {data_dir}')
        return pd.DataFrame()

    records = []
    for fp in sorted(files):
        try:
            df_station = load_station_file(fp)
            if is_ps2:
                # For PS2, latitude column is -9999; mark as NaN
                df_station['latitude'] = np.nan
            feats = engineer_features(df_station)
            feats['source_file'] = os.path.basename(fp)
            records.append(feats)
        except Exception as e:
            print(f'  Error loading {fp}: {e}')

    result = pd.DataFrame(records)
    print(f'Loaded {len(result)} stations from {data_dir}')
    return result


# ── PS1 (training / labeled) ─────────────────────────────────────────────────
df_ps1 = load_all_stations(PS1_DIR, is_ps2=False)

# ── PS2 (prediction / unlabeled) ─────────────────────────────────────────────
df_ps2 = load_all_stations(PS2_DIR, is_ps2=True)

print(f'\nPS1 shape: {df_ps1.shape}')
print(f'PS2 shape: {df_ps2.shape}')

if 'latitude' in df_ps1.columns:
    print(f'PS1 latitude range: {df_ps1["latitude"].min():.2f} to {df_ps1["latitude"].max():.2f}')

## 5. Model Training & Validation

**Choice of Model — Random Forest Regressor:**
- Handles missing values (via mean imputation in pipeline) and non-linear relationships.
- Built-in feature importance for interpretability.
- Robust to outliers and does not require extensive hyperparameter tuning.
- A natural baseline before more complex approaches (Gradient Boosting, Neural Networks).

**Validation Strategy:** K-Fold cross-validation across stations. Leave-one-out is ideal when N_stations is small; standard 5-fold otherwise.

In [ ]:
# ── Prepare feature matrix ────────────────────────────────────────────────────
META_COLS = ['station_code', 'latitude', 'longitude', 'source_file']


def prepare_xy(df: pd.DataFrame):
    """
    Split DataFrame into feature matrix X and target y.
    Drops non-feature columns and rows with missing latitude (for PS1).
    """
    feature_cols = [c for c in df.columns if c not in META_COLS]
    df_model = df.dropna(subset=['latitude'])  # only labeled rows for training
    X = df_model[feature_cols].copy()
    y = df_model['latitude'].values
    return X, y, df_model['station_code'].values


X_train, y_train, station_ids = prepare_xy(df_ps1)

print(f'Training samples (stations): {len(X_train)}')
print(f'Feature count: {X_train.shape[1]}')

# ── Build sklearn Pipeline ────────────────────────────────────────────────────
# Imputer handles any remaining NaN from sparse monthly bins
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestRegressor(
        n_estimators=200,
        max_features='sqrt',       # helps avoid overfitting on many features
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# ── Cross-validation ──────────────────────────────────────────────────────────
# Use LeaveOneOut if small dataset, else 5-fold
n_stations = len(X_train)
cv = LeaveOneOut() if n_stations <= 30 else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores_mae  = -cross_val_score(pipeline, X_train, y_train, cv=cv,
                                   scoring='neg_mean_absolute_error', n_jobs=-1)
cv_scores_r2   =  cross_val_score(pipeline, X_train, y_train, cv=cv,
                                   scoring='r2', n_jobs=-1)

print(f'\nCross-Validation Results ({type(cv).__name__}):')
print(f'  MAE  : {cv_scores_mae.mean():.3f} ± {cv_scores_mae.std():.3f} degrees')
print(f'  R²   : {cv_scores_r2.mean():.3f} ± {cv_scores_r2.std():.3f}')

# ── Fit on full training set ──────────────────────────────────────────────────
pipeline.fit(X_train, y_train)
y_pred_train = pipeline.predict(X_train)

print(f'\nIn-sample MAE : {mean_absolute_error(y_train, y_pred_train):.3f} degrees')
print(f'In-sample R²  : {r2_score(y_train, y_pred_train):.3f}')

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
rf_model      = pipeline.named_steps['model']
feature_names = X_train.columns.tolist()
importances   = pd.Series(rf_model.feature_importances_, index=feature_names)
top20         = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 5))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='k')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

print('Top 5 features:')
print(top20.head().to_string())

In [ ]:
# ── Residual analysis ─────────────────────────────────────────────────────────
if n_stations > 1:
    # Collect OOF predictions via cross-validation for unbiased residuals
    from sklearn.model_selection import cross_val_predict
    y_oof = cross_val_predict(pipeline, X_train, y_train, cv=cv, n_jobs=-1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Predicted vs Actual
    axes[0].scatter(y_train, y_oof, alpha=0.7, edgecolors='k', linewidths=0.5)
    lo, hi = y_train.min(), y_train.max()
    axes[0].plot([lo, hi], [lo, hi], 'r--', label='Perfect fit')
    axes[0].set_xlabel('Actual Latitude (°)')
    axes[0].set_ylabel('Predicted Latitude (°)')
    axes[0].set_title('Predicted vs Actual (OOF)')
    axes[0].legend()

    # Residuals
    residuals = y_oof - y_train
    axes[1].scatter(y_train, residuals, alpha=0.7, edgecolors='k', linewidths=0.5)
    axes[1].axhline(0, color='r', linestyle='--')
    axes[1].set_xlabel('Actual Latitude (°)')
    axes[1].set_ylabel('Residual (°)')
    axes[1].set_title('Residuals vs Actual Latitude')

    plt.tight_layout()
    plt.savefig('residual_analysis.png', dpi=120)
    plt.show()

## 6. Anomaly Detection — Identifying Out-of-Distribution (OOD) Stations

**Goal:** Flag PS2 stations that are *geographically novel* (i.e., outside the region of PS1 training data).

**Method — Mahalanobis Distance:**  
The Mahalanobis distance measures how many standard deviations a test point is from the training distribution, accounting for feature correlations. It is a classical, well-understood method for multivariate outlier detection (Mahalanobis, 1936).  

We compute it on the top-N most important features (rather than all ~100) to avoid the curse of dimensionality and covariance rank issues. A χ² threshold at a chosen confidence level (e.g., 99%) provides a principled decision boundary.

**Alternative approaches** (given more time):
- Isolation Forest (Liu et al., 2008) — non-parametric, no Gaussian assumption.
- One-class SVM.
- Conformal prediction intervals on model output.

In [ ]:
from scipy.stats import chi2


def detect_ood_stations(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    top_n_features: int = 20,
    confidence: float = 0.99
) -> pd.Series:
    """
    Compute Mahalanobis distance of each test station from the training
    distribution and flag those beyond the chi-squared threshold.

    Parameters
    ----------
    X_train        : Feature matrix used for training (imputed, scaled)
    X_test         : Feature matrix for PS2 stations (same columns)
    top_n_features : Use only the top-N most important features (avoids rank issues)
    confidence     : Chi-squared confidence level for OOD threshold

    Returns
    -------
    Series of Mahalanobis distances (index = X_test.index)
    """
    # Select top features by importance
    top_feats = importances.nlargest(top_n_features).index.tolist()

    # Impute + scale
    imputer = SimpleImputer(strategy='median')
    scaler  = StandardScaler()

    X_tr = scaler.fit_transform(imputer.fit_transform(X_train[top_feats]))
    X_te = scaler.transform(imputer.transform(X_test[top_feats]))

    # Compute covariance on training data
    cov = np.cov(X_tr.T)

    # Regularize covariance to avoid singularity (especially with many features)
    cov += np.eye(cov.shape[0]) * 1e-6
    cov_inv = np.linalg.inv(cov)

    train_mean = X_tr.mean(axis=0)

    distances = np.array([
        mahalanobis(row, train_mean, cov_inv)
        for row in X_te
    ])

    # Chi-squared threshold: k degrees of freedom = number of features
    k = top_n_features
    threshold = np.sqrt(chi2.ppf(confidence, df=k))
    print(f'Mahalanobis OOD threshold (χ² {confidence*100:.0f}%, k={k}): {threshold:.2f}')

    return pd.Series(distances, index=X_test.index), threshold


# ── Prepare PS2 feature matrix ────────────────────────────────────────────────
if not df_ps2.empty:
    feature_cols = [c for c in df_ps1.columns if c not in META_COLS]
    X_ps2 = df_ps2[feature_cols].copy()

    # Predict latitudes
    lat_predictions = pipeline.predict(X_ps2)

    # Detect OOD
    maha_distances, ood_threshold = detect_ood_stations(X_train, X_ps2)

    df_ps2['predicted_latitude'] = lat_predictions
    df_ps2['mahalanobis_dist']   = maha_distances.values
    df_ps2['is_ood']             = df_ps2['mahalanobis_dist'] > ood_threshold

    print(f'\nPS2 stations: {len(df_ps2)}')
    print(f'Flagged as OOD (new geography): {df_ps2["is_ood"].sum()}')
    print('\nPrediction summary:')
    print(df_ps2[['station_code','predicted_latitude','mahalanobis_dist','is_ood']].to_string())
else:
    print('No PS2 data loaded — running demo on PS1 data.')

## 7. Results Visualization

In [ ]:
if not df_ps2.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Predicted latitude distribution ──────────────────────────────────────
    colors = df_ps2['is_ood'].map({True: 'red', False: 'steelblue'})
    axes[0].scatter(
        df_ps2['station_code'].factorize()[0],
        df_ps2['predicted_latitude'],
        c=colors, s=80, edgecolors='k', linewidths=0.5
    )
    # Overlay PS1 training range
    axes[0].axhspan(y_train.min(), y_train.max(), alpha=0.1, color='green',
                    label='PS1 latitude range')
    axes[0].set_xlabel('Station index')
    axes[0].set_ylabel('Predicted Latitude (°)')
    axes[0].set_title('Predicted Latitudes (red = OOD)')
    axes[0].legend(handles=[
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',
                   markersize=8, label='In-distribution'),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
                   markersize=8, label='OOD (new geography)'),
    ])

    # ── Mahalanobis distances ─────────────────────────────────────────────────
    axes[1].bar(
        range(len(df_ps2)),
        df_ps2['mahalanobis_dist'],
        color=colors, edgecolor='k', linewidth=0.5
    )
    axes[1].axhline(ood_threshold, color='black', linestyle='--',
                    linewidth=2, label=f'OOD threshold ({ood_threshold:.1f})')
    axes[1].set_xlabel('Station index')
    axes[1].set_ylabel('Mahalanobis Distance')
    axes[1].set_title('OOD Detection: Mahalanobis Distance')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('ood_detection_results.png', dpi=120)
    plt.show()

## 8. Save Prediction Results

In [ ]:
if not df_ps2.empty:
    output_df = df_ps2[['station_code', 'source_file',
                         'predicted_latitude', 'mahalanobis_dist', 'is_ood']].copy()
    output_df.to_csv(OUTPUT_FILE, index=False)
    print(f'Results saved to {OUTPUT_FILE}')
    print(output_df.to_string())
else:
    # Demo output from the single sample station
    demo_df = pd.DataFrame([{
        'station_code':        df_sample['station_code'].iloc[0],
        'source_file':         'file_2002-2020_0_03502.csv',
        'actual_latitude':     df_sample['latitude'].iloc[0],
        'predicted_latitude':  float(pipeline.predict(
            X_train.iloc[[0]]
        )[0]) if len(X_train) > 0 else np.nan,
        'mahalanobis_dist':    0.0,
        'is_ood':              False,
    }])
    demo_df.to_csv(OUTPUT_FILE, index=False)
    print(f'Demo results saved to {OUTPUT_FILE}')
    print(demo_df.to_string())

print('\nDone!')

## 9. Discussion, Limitations & Next Steps

### What Works Well
- **Seasonal temperature swing** (`temp_seasonal_range`, `temp_seasonal_std`) are strong latitude proxies — near-equatorial stations show near-zero seasonality; mid-latitude stations show large swings.
- **Monthly temperature and radiation profiles** encode the insolation pattern driven by sun angle, directly related to latitude.
- **Random Forest** is robust to the high fraction of missing data (many -9999 values replaced with NaN) and does not require distributional assumptions.

### Limitations
1. **Sample size**: With few stations, the model may overfit. LOO-CV mitigates but does not eliminate this.
2. **Temporal coverage variation**: Stations cover different date ranges; monthly bins may be sparsely populated for short time series, adding noise.
3. **Hemisphere ambiguity**: A station at +20° and -20° may have mirror-image seasonal patterns (hot summer vs. cold summer). Without knowing the hemisphere, the model could confuse them. Incorporating the sign of the seasonal peak relative to the Northern Hemisphere summer (June–August) resolves this.
4. **OOD detection sensitivity**: Mahalanobis distance assumes Gaussian training distribution. With small n, the covariance estimate is noisy. Isolation Forest would be more robust.

### Next Steps (Given More Time)
1. **More data**: Ingest all available PS1 station files to increase training set size.
2. **Gradient Boosting** (LightGBM / XGBoost): Typically outperforms Random Forest on tabular data; can handle missing values natively.
3. **Hemisphere encoding**: Add a feature for which month has peak temperature to distinguish northern vs. southern hemisphere.
4. **Isotropy correction**: Longitude-aware features (e.g., distance from coast, elevation) could reduce residuals in edge cases.
5. **Conformal prediction**: Provide calibrated uncertainty intervals on each latitude prediction, not just point estimates.
6. **Isolation Forest OOD**: More robust to non-Gaussian training distributions.

### References
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12, 2825–2830.
- Mahalanobis, P.C. (1936). *On the generalised distance in statistics*. Proceedings of the National Institute of Sciences of India.
- Liu, F.T., Ting, K.M., Zhou, Z-H. (2008). *Isolation Forest*. ICDM 2008.
- INMET Brazilian Weather Station Dataset — Instituto Nacional de Meteorologia.